In [1]:
import os 
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_mcp_adapters.sessions import StdioConnection
from langchain_openai import ChatOpenAI

In [2]:
import nest_asyncio

nest_asyncio.apply()

In [3]:
import subprocess
import mcp.client.stdio as mcp_stdio

# Parche para notebooks en Windows: evita usar stderr con fileno no soportado
_orig_create = mcp_stdio.create_windows_process

async def _patched_create_windows_process(command, args, env=None, errlog=None, cwd=None):
    try:
        return await _orig_create(command, args, env=env, errlog=None, cwd=cwd)
    except Exception:
        # fallback seguro
        return await mcp_stdio._create_windows_fallback_process(
            command, args, env=env, errlog=subprocess.DEVNULL, cwd=cwd
        )

mcp_stdio.create_windows_process = _patched_create_windows_process


In [4]:
mcp_client = MultiServerMCPClient(
    {
        "find_healthcare_providers": StdioConnection(
            transport="stdio",
            command="uv",
            args=["run", "mcpserver.py"],
        )
    }
)
# tools = asyncio.run(mcp_client.get_tools())
tools = await mcp_client.get_tools()

In [5]:
agent = create_agent(
    ChatOpenAI(
        model="gpt-4.1-mini",
        openai_api_key=os.getenv("OPENAI_API_KEY"),
        openai_api_base=os.getenv("OPENAI_BASE_URL")
    ),
    tools,
    name="HealthcareProviderAgent",
    system_prompt="""Your task is to find and list healthcare providers 
    using the find_healthcare_providers MCP Tool based on the users query. 
    Only use providers based on the response from the tool.""",
)

In [6]:
prompt = "I'm based in Austin, TX. Are there any Psychiatrists near me?"
response = await agent.ainvoke(
    {
        "messages": [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    }
)

In [7]:
display(Markdown(response["messages"][-1].content))

There is a psychiatrist available near you in Austin, TX:

Dr. Jessica Coffey
- Specialty: Psychiatry
- Address: 1201 West 38th Street, Suite 210, Austin, TX 78705
- Phone: (512) 555-0199
- Email: j.coffey@austinmindhealth.com
- Years of Experience: 13
- Board Certified: Yes
- Hospital Affiliations: Ascension Seton Medical Center, St. David's Medical Center
- Education: Medical School at UT Southwestern Medical School, Residency at Dell Medical School
- Languages Spoken: English
- Accepts New Patients: Yes
- Insurance Accepted: Blue Cross Blue Shield, UnitedHealth, Aetna, Humana

Would you like assistance contacting Dr. Coffey or finding more psychiatrists?

In [8]:
from provider_agent import ProviderAgent

agent = await ProviderAgent().initialize()
result = await agent.answer_query(
    "I'm based in Austin, TX. Are there any Psychiatrists near me?"
)

display(Markdown(result))

Here is a psychiatrist in Austin, TX:

| Name             | Specialty  | Address                        | Phone         | Email                    | Years of Experience | Board Certified | Accepts New Patients | Insurance Accepted                        |
|------------------|------------|--------------------------------|---------------|--------------------------|---------------------|-----------------|---------------------|-------------------------------------------|
| Dr. Jessica Coffey | Psychiatry | 1201 West 38th Street, Suite 210, Austin, TX 78705 | (512) 555-0199 | j.coffey@austinmindhealth.com | 13                  | Yes             | Yes                 | Blue Cross Blue Shield, UnitedHealth, Aetna, Humana |

Would you like me to help you schedule an appointment or provide information about more psychiatrists nearby?